<a href="https://colab.research.google.com/github/profliuhao/CSIT599/blob/main/CSIT599_Online_lab1_neural_network_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Build a Neural Network from Scratch (NumPy)

**Module 1: Neural Networks Fundamentals**

In this lab you will build and train a small neural network **from scratch** using only
NumPy, then train it to solve a simple **binary classification** task.

The network is a classic 2-layer multi-layer perceptron (MLP):

```
input (2 features) --> [Linear] --> [ReLU] --> [Linear] --> [Sigmoid] --> P(class = 1)
```

## What you will do
You only need to fill in **four** short pieces of code, each marked with a `TODO`:
  1. the **ReLU** activation (and its derivative)
  2. the **forward pass**
  3. the **loss** (binary cross-entropy)
  4. the **backward pass** (the gradients)

Everything else — the data, weight initialization, the **optimizers**, the training loop,
evaluation, and the plots — is already written for you.

## How to use this file
* Fill in each `TODO`, then run the cells from top to bottom (or "Run All").
* Use the **gradient check** cell to confirm your backward pass is correct.
* To compare optimizers, change a **single line** in the Config cell:
  `OPTIMIZER = "adam"` -> try `"sgd"`, `"momentum"`, or `"rmsprop"`.
  (You will analyze that comparison in the M1 Discussion — no need to write it up here.)

## Setup — imports and configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ----------------------------- Config -----------------------------
SEED = 42            # makes results reproducible
HIDDEN_SIZE = 16     # number of neurons in the hidden layer
EPOCHS = 1500        # number of full passes over the training data
PRINT_EVERY = 150    # how often to print progress
SHOW_PLOTS = True    # set False to skip the matplotlib figures

# ---- Optimizer: change ONE line to switch between optimizers ----
OPTIMIZER = "adam"   # options: "sgd" | "momentum" | "rmsprop" | "adam"

# A sensible learning rate is pre-chosen for each optimizer so switching "just works".
LEARNING_RATES = {"sgd": 0.5, "momentum": 0.3, "rmsprop": 0.02, "adam": 0.05}
LEARNING_RATE = LEARNING_RATES[OPTIMIZER]
# -------------------------------------------------------------------

np.random.seed(SEED)
print(f"Using optimizer = '{OPTIMIZER}' with learning rate = {LEARNING_RATE}")

## Section 1 — The data  *(provided)*

We use a small 2-D "two moons" dataset: two interleaving half-circles that are **not**
linearly separable, so the hidden layer actually has to do some work. Features are
standardized (mean 0, std 1), which helps training.

In [ ]:
# Create the dataset (400 points, a little noise so it is not trivial).
X, y = make_moons(n_samples=400, noise=0.20, random_state=SEED)

# Split into train / test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

# Standardize features using statistics from the TRAINING set only.
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# Reshape labels to column vectors of shape (m, 1) and use float for the math.
y_train = y_train.reshape(-1, 1).astype(float)
y_test = y_test.reshape(-1, 1).astype(float)

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test :", X_test.shape, "  y_test :", y_test.shape)

In [ ]:
# Quick look at the data (provided).
if SHOW_PLOTS:
    plt.figure(figsize=(5, 4))
    plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train.ravel(), cmap="RdBu", edgecolors="k", s=20)
    plt.title("Training data (two moons)")
    plt.xlabel("feature 1")
    plt.ylabel("feature 2")
    plt.show()

## Section 2 — Activation functions

`sigmoid` is given. **TODO 1:** implement `relu` and its derivative `relu_deriv`.

In [ ]:
def sigmoid(z):
    """Sigmoid activation (given). Maps any real number to (0, 1)."""
    z = np.clip(z, -500, 500)          # clip for numerical stability (avoids overflow)
    return 1.0 / (1.0 + np.exp(-z))


def relu(z):
    """ReLU activation: returns z where z > 0, and 0 otherwise."""
    # TODO 1a: return the elementwise maximum of 0 and z.
    #          Hint: np.maximum(0.0, z)
    raise NotImplementedError("TODO 1a: implement relu, then delete this line.")


def relu_deriv(z):
    """Derivative of ReLU with respect to its input: 1 where z > 0, else 0."""
    # TODO 1b: return an array that is 1.0 where z > 0 and 0.0 elsewhere.
    #          Hint: (z > 0).astype(float)
    raise NotImplementedError("TODO 1b: implement relu_deriv, then delete this line.")

## Section 3 — Weight initialization  *(provided)*

Weights get small random values (He initialization, which works well with ReLU) and
biases start at zero. `params` is a dictionary holding the two weight matrices and
two bias vectors.

In [ ]:
def init_parameters(n_in, n_hidden, n_out, seed=SEED):
    """Return a dict of parameters: W1, b1 (layer 1) and W2, b2 (layer 2)."""
    rng = np.random.default_rng(seed)
    params = {
        "W1": rng.standard_normal((n_in, n_hidden)) * np.sqrt(2.0 / n_in),
        "b1": np.zeros((1, n_hidden)),
        "W2": rng.standard_normal((n_hidden, n_out)) * np.sqrt(2.0 / n_hidden),
        "b2": np.zeros((1, n_out)),
    }
    return params

## Section 4 — Forward propagation

**TODO 2:** compute the network's output. With `m` examples and the shapes shown,
the four quantities are:

| quantity | formula | shape |
|----------|---------|-------|
| `Z1` | `X @ W1 + b1` | (m, hidden) |
| `A1` | `relu(Z1)`    | (m, hidden) |
| `Z2` | `A1 @ W2 + b2`| (m, 1) |
| `A2` | `sigmoid(Z2)` | (m, 1) |

`@` is matrix multiplication; the bias rows broadcast over all `m` examples.
We keep the intermediate values in `cache` because the backward pass needs them.

In [ ]:
def forward(X, params):
    """Run a forward pass. Returns A2 (predicted probabilities) and a cache of values."""
    W1, b1, W2, b2 = params["W1"], params["b1"], params["W2"], params["b2"]

    # ===== TODO 2: define Z1, A1, Z2, A2 using the table above =====
    raise NotImplementedError("TODO 2: implement the forward pass (define Z1, A1, Z2, A2), then delete this line.")
    # ===============================================================

    cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

## Section 5 — Loss: binary cross-entropy

**TODO 3:** implement the average binary cross-entropy over the `m` examples:

`loss = -(1/m) * sum( y * log(A2) + (1 - y) * log(1 - A2) )`

Add a tiny `eps` inside the logs so we never take `log(0)`.

In [ ]:
def compute_loss(A2, y):
    """Average binary cross-entropy between predictions A2 and true labels y."""
    m = y.shape[0]
    eps = 1e-8  # keeps log() finite

    # ===== TODO 3: compute the binary cross-entropy loss =====
    raise NotImplementedError("TODO 3: implement binary cross-entropy, then delete this line.")
    # =========================================================

    return loss

## Section 6 — Backpropagation

**TODO 4:** compute the gradient of the loss with respect to every parameter.
Work backward from the output. A convenient fact: for a **sigmoid output combined with
binary cross-entropy**, the output error simplifies to `dZ2 = A2 - y`.

```
dZ2 = A2 - y                                   # (m, 1)
dW2 = (1/m) * A1.T @ dZ2                        # (hidden, 1)
db2 = (1/m) * sum(dZ2, axis=0, keepdims=True)   # (1, 1)

dA1 = dZ2 @ W2.T                                # (m, hidden)
dZ1 = dA1 * relu_deriv(Z1)                      # (m, hidden)   <- chain rule through ReLU
dW1 = (1/m) * X.T @ dZ1                         # (n_features, hidden)
db1 = (1/m) * sum(dZ1, axis=0, keepdims=True)   # (1, hidden)
```

In [ ]:
def backward(params, cache, y):
    """Return a dict of gradients with the same keys as params: W1, b1, W2, b2."""
    m = y.shape[0]
    W2 = params["W2"]
    X, Z1, A1, A2 = cache["X"], cache["Z1"], cache["A1"], cache["A2"]

    # ===== TODO 4: compute dZ2, dW2, db2, dA1, dZ1, dW1, db1 using the formulas above =====
    raise NotImplementedError("TODO 4: implement backpropagation (define dW1, db1, dW2, db2), then delete this line.")
    # =====================================================================================

    grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
    return grads

## Section 7 — Optimizers  *(provided — you do not need to edit this)*

One small class implements four common optimizers. They all expose the same `step()`
method, which updates the parameters in place given the gradients. Switch between them
from the Config cell (`OPTIMIZER = ...`). You will compare their behavior in the
discussion — here they are just a tool you can swap freely.

In [ ]:
class Optimizer:
    """SGD, SGD+Momentum, RMSProp, and Adam behind one simple interface."""

    def __init__(self, kind="adam", lr=0.05, momentum=0.9, beta1=0.9, beta2=0.999, eps=1e-8):
        self.kind = kind.lower()
        self.lr = lr
        self.momentum = momentum
        self.beta1, self.beta2, self.eps = beta1, beta2, eps
        self._state = {}   # per-parameter running averages (used by momentum/rmsprop/adam)
        self._t = 0        # timestep counter (used by Adam's bias correction)

    def step(self, params, grads):
        """Update each parameter in `params` in place using its gradient in `grads`."""
        self._t += 1
        for key in params:
            g = grads[key]

            if self.kind == "sgd":
                # Plain gradient descent: step opposite the gradient.
                params[key] -= self.lr * g

            elif self.kind == "momentum":
                # Keep a velocity that accumulates past gradients (smooths the path).
                v = self._state.setdefault(key, np.zeros_like(g))
                v[...] = self.momentum * v - self.lr * g
                params[key] += v

            elif self.kind == "rmsprop":
                # Scale the step by a running average of squared gradients.
                s = self._state.setdefault(key, np.zeros_like(g))
                s[...] = self.beta2 * s + (1 - self.beta2) * (g ** 2)
                params[key] -= self.lr * g / (np.sqrt(s) + self.eps)

            elif self.kind == "adam":
                # Combine momentum (m) and RMSProp-style scaling (v), with bias correction.
                st = self._state.setdefault(key, {"m": np.zeros_like(g), "v": np.zeros_like(g)})
                st["m"] = self.beta1 * st["m"] + (1 - self.beta1) * g
                st["v"] = self.beta2 * st["v"] + (1 - self.beta2) * (g ** 2)
                m_hat = st["m"] / (1 - self.beta1 ** self._t)
                v_hat = st["v"] / (1 - self.beta2 ** self._t)
                params[key] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

            else:
                raise ValueError(f"Unknown optimizer: '{self.kind}'")

## Section 8 — Helpers: accuracy & predictions  *(provided)*

In [ ]:
def accuracy(A2, y):
    """Fraction of correct predictions (threshold the probability at 0.5)."""
    preds = (A2 >= 0.5).astype(float)
    return float(np.mean(preds == y))


def predict(X, params):
    """Return hard 0/1 class predictions for inputs X."""
    A2, _ = forward(X, params)
    return (A2 >= 0.5).astype(int)

## Section 9 — Gradient check  *(provided, optional but recommended)*

This numerically estimates each gradient by nudging each parameter a tiny bit and seeing
how the loss changes, then compares that estimate to your backward pass. If your TODO 4
is correct, every relative error should be tiny (about 1e-7 or smaller).

In [ ]:
def gradient_check(params, X, y, eps=1e-6):
    """Compare analytic gradients (backward) to numerical gradients. Returns rel. errors."""
    _, cache = forward(X, params)
    grads = backward(params, cache, y)

    rel_errors = {}
    for key in params:
        numeric = np.zeros_like(params[key])
        it = np.nditer(params[key], flags=["multi_index"])
        while not it.finished:
            idx = it.multi_index
            original = params[key][idx]

            params[key][idx] = original + eps
            loss_plus = compute_loss(forward(X, params)[0], y)
            params[key][idx] = original - eps
            loss_minus = compute_loss(forward(X, params)[0], y)
            params[key][idx] = original  # restore

            numeric[idx] = (loss_plus - loss_minus) / (2 * eps)
            it.iternext()

        denom = np.linalg.norm(numeric) + np.linalg.norm(grads[key]) + 1e-12
        rel_errors[key] = float(np.linalg.norm(numeric - grads[key]) / denom)
    return rel_errors


# Run a check on a tiny network so it is fast. (Requires TODO 1-4 to be implemented.)
_check_params = init_parameters(n_in=X_train.shape[1], n_hidden=5, n_out=1, seed=0)
_errors = gradient_check(_check_params, X_train[:20], y_train[:20])
print("Gradient check relative errors (want ~1e-7 or smaller):")
for k, v in _errors.items():
    print(f"  {k}: {v:.2e}")

## Section 10 — Training loop  *(provided)*

Each epoch does the four steps you implemented — forward, loss, backward — and then asks
the optimizer to update the parameters. We record the loss and accuracy so we can plot them.

In [ ]:
def train(X, y, params, optimizer, epochs=EPOCHS, print_every=PRINT_EVERY):
    history = {"loss": [], "acc": []}
    for epoch in range(1, epochs + 1):
        A2, cache = forward(X, params)        # your TODO 2
        loss = compute_loss(A2, y)            # your TODO 3
        grads = backward(params, cache, y)    # your TODO 4
        optimizer.step(params, grads)         # provided

        history["loss"].append(loss)
        history["acc"].append(accuracy(A2, y))
        if epoch == 1 or epoch % print_every == 0:
            print(f"epoch {epoch:4d} | loss {loss:.4f} | train acc {history['acc'][-1]:.3f}")
    return history

In [ ]:
# Build a fresh network and train it with the optimizer chosen in the Config cell.
params = init_parameters(n_in=X_train.shape[1], n_hidden=HIDDEN_SIZE, n_out=1, seed=SEED)
optimizer = Optimizer(kind=OPTIMIZER, lr=LEARNING_RATE)
history = train(X_train, y_train, params, optimizer)

## Section 11 — Evaluate & visualize  *(provided)*

In [ ]:
train_acc = accuracy(forward(X_train, params)[0], y_train)
test_acc = accuracy(forward(X_test, params)[0], y_test)
print(f"Final train accuracy: {train_acc:.3f}")
print(f"Final test  accuracy: {test_acc:.3f}")

In [ ]:
if SHOW_PLOTS:
    plt.figure(figsize=(5, 4))
    plt.plot(history["loss"])
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(f"Training loss ({OPTIMIZER})")
    plt.show()

In [ ]:
def plot_decision_boundary(X, y, params, title="Decision boundary"):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    probs = forward(np.c_[xx.ravel(), yy.ravel()], params)[0].reshape(xx.shape)

    plt.figure(figsize=(5, 4))
    plt.contourf(xx, yy, probs, levels=20, cmap="RdBu", alpha=0.6)
    plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="RdBu", edgecolors="k", s=20)
    plt.title(title)
    plt.xlabel("feature 1")
    plt.ylabel("feature 2")
    plt.show()


if SHOW_PLOTS:
    plot_decision_boundary(X_test, y_test, params, title=f"Decision boundary ({OPTIMIZER})")

## Section 12 — Try it yourself  *(optional exploration)*

Once everything runs, experiment by re-running the notebook after changing one thing at a
time. These are just suggestions to build intuition — there is nothing to submit here.

* **Switch optimizers:** in the Config cell set `OPTIMIZER` to `"sgd"`, `"momentum"`,
  `"rmsprop"`, or `"adam"` and compare the loss curves. (You will discuss this in the
  M1 Discussion.)
* **Learning rate:** try a much smaller or larger value and watch how the loss curve changes.
* **Capacity:** change `HIDDEN_SIZE` (e.g., 2, 8, 64) and see how the decision boundary changes.
* **Harder data:** increase the `noise` in `make_moons` and see how accuracy responds.